In [ ]:
%matplotlib qt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.transforms import Bbox
import mne, os, pickle, csv, json
from pyxdf import load_xdf
import pandas as pd

In [ ]:
# Create output folder
out_dir = 'artifact_results'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [ ]:
# Input files
input_dir = r'S:\laura-wheeler_in-ear-eeg-auditory-bci_0589_data_prism\Raw Data\Study Data Organized - in-ear EEG and scalp working'

# Find all xdf files
xdf_files = {}
participant_folders = [f for f in os.listdir(input_dir) if f.startswith('Participant') and f[-1].isdigit()]
for participant_folder in participant_folders:
    xdf_files[participant_folder] = [f for f in os.listdir(os.path.join(input_dir, participant_folder, 'sourcedata')) if 'blink' in f and f.endswith('.xdf')]

xdf_files

In [ ]:
# Load EEG
epochs = {}
for participant in xdf_files:
    print('Parasing XDF files for %s' % participant)
    
    streams = {}
    for f in range(len(xdf_files[participant])):
        xdf_file = xdf_files[participant][f]
        print('file: ', xdf_file)
        
        # Load LSL streams from xdf
        found_streams, _ = load_xdf(os.path.join(input_dir, participant, 'sourcedata', xdf_file))
        for found_stream in found_streams:
            stream_name = found_stream['info']['name'][0]
            if stream_name not in streams:
                streams[stream_name] = []
            streams[stream_name].append(found_stream)
            
    # Create Raw data
    eeg_raw_list, raw = {}, {}
    for stream in ['BrainAmpSeries-Dev_1', 'InEarEEG']:
        # Load EEG data
        for i in range(len(streams[stream])):
            print('Processing %s stream' % stream)
            # Get EEG info
            if stream == 'BrainAmpSeries-Dev_1':
                sfreq = float(streams[stream][i]['info']['nominal_srate'][0])
                ch_names = [ch['label'][0] for ch in streams[stream][i]['info']['desc'][0]['channels'][0]['channel']]
            elif stream == 'InEarEEG':
                sfreq = float(streams[stream][i]['info']['effective_srate'])
                ch_names = ['A1', 'A2']
            eeg_info = mne.create_info(ch_names, sfreq, ch_types='eeg')
            montage = mne.channels.read_custom_montage('%s_montage.elc' % stream)

            # Apply scaling factor to EEG data
            if stream == 'InEarEEG':
                eeg_data = np.transpose(streams[stream][i]['time_series']) / 24 / 4.1887e9 * 4.5
            elif stream == 'BrainAmpSeries-Dev_1':
                eeg_data = np.transpose(streams[stream][i]['time_series']) / 1e6
            eeg_raw = mne.io.RawArray(eeg_data, eeg_info, verbose=False)
            eeg_raw.set_montage(montage)

            # marker annotations
            if stream == 'BrainAmpSeries-Dev_1':
                # detect onsets of blinks
                markers = streams['mindset_Marker_PRISM-DIY-MSI-4'][i]['time_series']
                markers_ts = streams['mindset_Marker_PRISM-DIY-MSI-4'][i]['time_stamps']
                eog_ch = eeg_raw.copy().pick('Fz').filter(l_freq=1, h_freq=10, verbose=False).get_data()
                thresh = (np.max(eog_ch) - np.min(eog_ch)) * 0.5
                eog_events = mne.preprocessing.find_eog_events(eeg_raw, ch_name='Fz', tstart=markers_ts[-1] - streams[stream][i]['time_stamps'][0], thresh=thresh, filter_length='3s', verbose=False)
                onsets, durations, descriptions = [], [], []
                scalp_ear_offset = streams['InEarEEG'][i]['time_stamps'][0] - streams['BrainAmpSeries-Dev_1'][i]['time_stamps'][0]
                for event in eog_events:
                    onset = event[0] / eeg_raw.info["sfreq"]
                    if onset - 0.6 < (markers_ts[-1] - streams[stream][i]['time_stamps'][0]):
                        continue
                    elif onset + 0.6 > (streams[stream][i]['time_stamps'][-1] - streams[stream][i]['time_stamps'][0]):
                        continue
                    if onset - 0.6 - scalp_ear_offset < (markers_ts[-1] - streams['InEarEEG'][i]['time_stamps'][0]):
                        print('SKIPPED')
                        continue
                    elif onset + 0.6 - scalp_ear_offset > (streams['InEarEEG'][i]['time_stamps'][-1] - streams['InEarEEG'][i]['time_stamps'][0]):
                        print('SKIPPED')
                        continue
                    onsets.append(onset)
                    durations.append(0.4)
                    descriptions.append('blink')
                annotations = mne.Annotations(onsets, durations, descriptions)
                eeg_raw = eeg_raw.set_annotations(annotations)
            elif stream == 'InEarEEG':
                onsets, durations, descriptions = [], [], []
                annotations = eeg_raw_list['BrainAmpSeries-Dev_1'][i].annotations
                annotations.onset = annotations.onset - (streams['InEarEEG'][i]['time_stamps'][0] - streams['BrainAmpSeries-Dev_1'][i]['time_stamps'][0])
                eeg_raw = eeg_raw.set_annotations(annotations)
                eeg_raw = eeg_raw.resample(250)

            # Add raw to list
            if stream not in eeg_raw_list:
                eeg_raw_list[stream] = []
            eeg_raw_list[stream].append(eeg_raw)

        # Concatenate all epochs
        raw[stream] = mne.concatenate_raws(eeg_raw_list[stream])
        raw[stream] = raw[stream].filter(l_freq=1.0, h_freq=40.0)

        # Create epochs
        events, event_id = mne.events_from_annotations(raw[stream])
        if participant not in epochs:
            epochs[participant] = {}
        epochs[participant][stream] = mne.Epochs(
            raw[stream], events, event_id=event_id, preload=True,
            # tmin=-0.6, tmax=0.6, baseline=(-0.6, -0.4)
            tmin=-1, tmax=1, baseline=(-0.6, -0.4)
        )

In [ ]:
epochs

## Plot ERP

In [ ]:
plt.close('all')
fig, ax = plt.subplots(4,2, figsize=(8,8))
title = 'Blink Artifact Event Related Potentials'
fig.suptitle(title)

for i in range(len(ax)):
    participant = list(epochs.keys())[i]
    # Plot Scalp ERP
    evoked = epochs[participant]['BrainAmpSeries-Dev_1'].average().crop(tmin=-0.6, tmax=0.6)
    # evoked = epochs[participant]['BrainAmpSeries-Dev_1'].average().crop(tmin=-1, tmax=1)
    evoked.plot(axes=ax[i,0], picks='all')
    ax[i,0].set_title('')
    ax[i,0].set_ylabel('P%d\nµV' % (i+1))
    if i != len(ax)-1:
        ax[i,0].set_xlabel('')
        ax[i,0].set_xticks([-0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6], ['', '', '', '', '', '', ''])
    # Plot In-Ear ERP
    evoked = epochs[participant]['InEarEEG'].average().crop(tmin=-0.6, tmax=0.6)
    # evoked = epochs[participant]['InEarEEG'].average().crop(tmin=-1, tmax=1)
    evoked.plot(axes=ax[i,1])
    ax[i,1].set_title('')
    ax[i,1].set_ylabel('')
    if i != len(ax)-1:
        ax[i,1].set_xlabel('')
        ax[i,1].set_xticks([-0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6], ['', '', '', '', '', '', ''])
ax[0,0].set_title('Scalp EEG')
ax[0,1].set_title('In-Ear EEG')

# Adjust ERP y-axes
for j in range(2):
    ylim = np.array([ax[i,j].get_ylim() for i in range(len(ax))])
    ylim = [np.min(ylim[:,0]), np.max(ylim[:,1])]
    for i in range(len(ax)):
        ax[i,j].set_ylim(ylim)
        ax[i,j].vlines(0, ylim[0], ylim[1], linestyles='dashed', colors='gray')

# Adjust spacing
plt.subplots_adjust(left=0.11, right=0.98, top=0.91, bottom=0.07, wspace=0.125, hspace=0.21)

fig.savefig(os.path.join(out_dir, '%s.png'%title))
fig.show()

In [ ]:
# Compute amplitudes
clean_intervals = {
    'Participant 35': (-0.6, -0.35),
    'Participant 37': (-0.6, -0.42),
    'Participant 38': (-1, -0.6),
    'Participant 40': (-1, -0.72)
}

def compute_amplitudes(participant, headset, evoked, clean_interval=(-0.6, -0.45), artifact_interval=(-0.6, 0.6)):
    evoked_clean = evoked.get_data()[:,np.logical_and(evoked.times >= clean_interval[0], evoked.times <= clean_interval[1])]
    evoked_artifact = evoked.get_data()[:,np.logical_and(evoked.times >= artifact_interval[0], evoked.times <= artifact_interval[1])]
    amp_clean = (np.max(evoked_clean, axis=1) - np.min(evoked_clean, axis=1)) * 1e6
    amp_artifact = (np.max(evoked_artifact, axis=1) - np.min(evoked_artifact, axis=1)) * 1e6
    percent_change = (amp_artifact - amp_clean) / amp_clean * 100
    max_increase_idx = np.argmax(percent_change)
    max_amp_idx = np.argmax(amp_artifact)
    return {
        'Participant': participant.replace('Participant ',''),
        'Channels': headset,
        'Clean Amplitude': np.max(amp_clean),
        'Artifact Amplitude': np.max(amp_artifact),
        'Max Increase Channel': evoked.ch_names[max_increase_idx],
        'Max Percent Change': np.max(percent_change),
        'Max Amplitude Channel': evoked.ch_names[max_amp_idx],
        'Max Amp Percent Change': percent_change[max_amp_idx],
    }
    
amplitudes = []
for participant in clean_intervals:
    # Scalp
    evoked = epochs[participant]['BrainAmpSeries-Dev_1'].average().crop(tmin=-1, tmax=1)
    ampdict = compute_amplitudes(participant, 'Scalp', evoked, clean_interval=clean_intervals[participant])
    amplitudes.append(ampdict)
    
    # In-Ear
    evoked = epochs[participant]['InEarEEG'].average().crop(tmin=-1, tmax=1)
    ampdict = compute_amplitudes(participant, 'In-Ear', evoked, clean_interval=clean_intervals[participant])
    amplitudes.append(ampdict)

amplitudes = pd.DataFrame(amplitudes)
# amplitudes['Percent Change'] = ((amplitudes['Artifact Amplitude'] - amplitudes['Clean Amplitude']) / amplitudes['Clean Amplitude']) * 100
amplitudes

In [ ]:
amplitudes.loc[amplitudes['Channels'] == 'Scalp']

In [ ]:
amplitudes.loc[amplitudes['Channels'] == 'In-Ear']

In [ ]:
amplitudes.loc[amplitudes['Channels'] == 'Scalp']['Max Amp Percent Change'].mean()

In [ ]:
amplitudes.loc[amplitudes['Channels'] == 'In-Ear']['Percent Change'].mean()